# 因子分析
分析因子暴露、收益和换手率

In [ ]:
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import pandas as pd
import plotly.express as px
from helpers import get_experiments, get_best_strategies

try:
    experiments = get_experiments(limit=100)
    has_data = len(experiments) > 0
except Exception:
    experiments = pd.DataFrame()
    has_data = False

In [ ]:
FACTOR_LIBRARY = [
    "momentum_20d", "momentum_60d", "momentum_120d",
    "value_pe", "value_pb", "value_pc",
    "quality_roe", "quality_debt", "quality_growth",
    "low_vol_20d", "low_vol_60d",
    "size_log_mcap",
]

if has_data:
    # Count factor usage across all experiments
    factor_counts = {f: 0 for f in FACTOR_LIBRARY}
    for _, row in experiments.iterrows():
        strategy = row.get("strategy", {})
        if isinstance(strategy, dict):
            for f in strategy.get("factors", []):
                if f in factor_counts:
                    factor_counts[f] += 1

    df_factors = pd.DataFrame(
        list(factor_counts.items()), columns=["factor", "count"]
    )
    fig = px.bar(df_factors, x="factor", y="count", title="因子使用频率")
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

In [ ]:
if has_data:
    weight_data = []
    for _, row in experiments.iterrows():
        strategy = row.get("strategy", {})
        if isinstance(strategy, dict):
            for f, w in strategy.get("weights", {}).items():
                weight_data.append({"factor": f, "weight": w})

    if weight_data:
        df_weights = pd.DataFrame(weight_data)
        fig = px.box(df_weights, x="factor", y="weight", title="因子权重分布")
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()

In [ ]:
if has_data:
    try:
        best = get_best_strategies(top_k=10)
        if len(best) > 0:
            # Build heatmap data: factor presence in top strategies
            heatmap_data = {f: [] for f in FACTOR_LIBRARY}
            strategy_labels = []
            for idx, row in best.iterrows():
                strategy = row.get("strategy", {})
                if not isinstance(strategy, dict):
                    continue
                label = f"策略{idx + 1}"
                if "final_nav" in row:
                    label += f" (NAV {row['final_nav']:.2f})"
                strategy_labels.append(label)
                factors_in_strategy = strategy.get("factors", [])
                weights_map = strategy.get("weights", {})
                for f in FACTOR_LIBRARY:
                    heatmap_data[f].append(weights_map.get(f, 0))

            df_heatmap = pd.DataFrame(heatmap_data, index=strategy_labels)
            fig = px.imshow(
                df_heatmap.T,
                labels=dict(x="策略", y="因子", color="权重"),
                title="Top策略因子权重热力图",
                aspect="auto",
            )
            fig.show()
        else:
            print("暂无Top策略数据")
    except Exception as e:
        print(f"加载Top策略失败: {e}")

In [ ]:
if has_data:
    # Factor performance scatter: sharpe vs max_drawdown by factor
    perf_data = []
    for _, row in experiments.iterrows():
        strategy = row.get("strategy", {})
        result = row.get("result", {})
        if not isinstance(strategy, dict) or not isinstance(result, dict):
            continue
        sharpe = result.get("sharpe")
        max_dd = result.get("max_drawdown")
        if sharpe is None or max_dd is None:
            continue
        factors = strategy.get("factors", [])
        for f in factors:
            perf_data.append({"factor": f, "sharpe": sharpe, "max_drawdown": max_dd})

    if perf_data:
        df_perf = pd.DataFrame(perf_data)
        fig = px.scatter(
            df_perf,
            x="max_drawdown",
            y="sharpe",
            color="factor",
            title="因子表现: 夏普率 vs 最大回撤",
            labels={"max_drawdown": "最大回撤", "sharpe": "夏普率"},
        )
        fig.show()
    else:
        print("暂无因子表现数据")

In [ ]:
if not has_data:
    print("暂无实验数据")
    print("请先启动 internal-store MCP server:")
    print("  uvicorn mcp-servers.internal-store.server:mcp_app --port 8002")